# IMU IK to ID Comparison (JinWoo_MT_IK)

This notebook runs inference from IMU-derived IK angles using `runs_0507_all/best_model.pt` (with fallback to `runs/0507_ik_id_all/best_model.pt`) and compares model outputs against OpenSim inverse dynamics.

Pipeline used here:
- Angle input: **6 Hz causal low-pass**, 4th order
- Angular velocity input: **15 Hz causal low-pass**, 4th order
- Model output: **6 Hz causal low-pass**, 4th order

Because the MTw IK pickle files do not include timestamps, alignment is done by sample index over the shared trial duration.

In [ ]:
import sys
import warnings
import pickle
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfilt, sosfilt_zi
import ipywidgets as widgets
from IPython.display import display

warnings.filterwarnings("ignore", message=".*NumPy.*")

In [ ]:
# Paths
PROJECT_ROOT = Path('/Users/luorix/Desktop/MetaMobility Lab (CMU)/projects/MeMo-IK-ID')
IMU_IK_ROOT = Path('/Users/luorix/Downloads/JinWoo_MT_IK/outputs/mtw_ik')
OPENSIM_ROOT = Path('/Users/luorix/Desktop/MetaMobility Lab (CMU)/projects/opensim-processing/data')

# User-requested checkpoint path, with robust fallback to current repo structure
CHECKPOINT_CANDIDATES = [
    PROJECT_ROOT / 'runs_0507_all' / 'best_model.pt',
    PROJECT_ROOT / 'runs/0507_ik_id_all' / 'best_model.pt',
    PROJECT_ROOT / 'best_model.pt',
]
CHECKPOINT = next((p for p in CHECKPOINT_CANDIDATES if p.exists()), None)
if CHECKPOINT is None:
    raise FileNotFoundError(f'No checkpoint found in candidates: {CHECKPOINT_CANDIDATES}')

# Filter specs (requested)
ANGLE_CUTOFF_HZ = 6.0
VEL_CUTOFF_HZ = 15.0
OUT_CUTOFF_HZ = 6.0
FILTER_ORDER = 4

# Data assumptions
DEFAULT_FS_HZ = 100.0  # used when timestamps are unavailable (IMU IK pkl)
PREFERRED_EXO_FOLDER = 'awinda'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

sys.path.insert(0, str(PROJECT_ROOT))
from dataset import IK_DOF_NAMES
from model import TCN, TransformerMoment, GaussianDiffusion1D

print(f'Checkpoint: {CHECKPOINT}')
print(f'Device: {DEVICE}')

In [ ]:
def parse_opensim_table(path: Path) -> pd.DataFrame:
    with open(path, 'r') as f:
        header_end = None
        for i, line in enumerate(f):
            if line.strip().lower() == 'endheader':
                header_end = i
                break
    if header_end is None:
        raise RuntimeError(f'endheader not found: {path}')
    df = pd.read_csv(path, sep=r'\s+', skiprows=header_end + 1)
    return df.set_index('time')


def causal_lpf(x: np.ndarray, fs_hz: float, cutoff_hz: float, order: int = 4) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    if x.ndim != 1:
        raise ValueError('causal_lpf expects 1D input')
    nyq = 0.5 * fs_hz
    if cutoff_hz <= 0 or cutoff_hz >= nyq or len(x) < 4:
        return x.copy()
    sos = butter(order, cutoff_hz / nyq, btype='low', output='sos')
    zi = sosfilt_zi(sos) * x[0]
    y, _ = sosfilt(sos, x, zi=zi)
    return y


def causal_lpf_multichannel(X: np.ndarray, fs_hz: float, cutoff_hz: float, order: int = 4) -> np.ndarray:
    X = np.asarray(X, dtype=np.float64)
    if X.ndim != 2:
        raise ValueError('causal_lpf_multichannel expects (T, C)')
    return np.column_stack([causal_lpf(X[:, c], fs_hz=fs_hz, cutoff_hz=cutoff_hz, order=order) for c in range(X.shape[1])])


def estimate_subject_mass_kg(subject_dir: Path) -> float:
    opensim_dir = subject_dir / 'opensim'
    if not opensim_dir.exists():
        return np.nan
    no_exo = sorted(opensim_dir.glob('*no_exo.osim'))
    if not no_exo:
        return np.nan
    tree = ET.parse(no_exo[0])
    root = tree.getroot()
    masses = []
    for body in root.findall('.//Body'):
        m = body.find('mass')
        if m is not None and m.text:
            try:
                masses.append(float(m.text))
            except ValueError:
                pass
    if not masses:
        for body in root.findall('.//{*}Body'):
            m = body.find('{*}mass')
            if m is not None and m.text:
                try:
                    masses.append(float(m.text))
                except ValueError:
                    pass
    return float(np.sum(masses)) if masses else np.nan


def filename_to_condition(stem: str) -> str:
    speed_token, cond_token = stem.split('_', 1)  # e.g. 0p8mps_lg
    return f'{cond_token.upper()}_{speed_token}'   # -> LG_0p8mps


def find_id_path(subject: str, condition: str) -> tuple[Path | None, str | None]:
    subject_dir = OPENSIM_ROOT / subject
    exo_order = [PREFERRED_EXO_FOLDER, 'hip-exo', 'knee-exo']
    for exo in exo_order:
        p = subject_dir / exo / 'id' / f'{condition}_id.sto'
        if p.exists():
            return p, exo
    return None, None


print('Helpers defined.')

In [ ]:
# Load checkpoint model + metadata
ckpt = torch.load(str(CHECKPOINT), map_location=DEVICE, weights_only=False)
cfg = ckpt['model_config']
model_type = cfg.get('model_type', 'tcn')

if model_type == 'transformer':
    model = TransformerMoment(
        n_input_channels=cfg['n_input_channels'],
        n_output_channels=cfg['n_output_channels'],
        d_model=cfg['d_model'],
        n_heads=cfg['n_heads'],
        n_layers=cfg['n_layers'],
        d_ff=cfg['d_ff'],
        dropout=cfg['dropout'],
    )
elif model_type == 'diffusion':
    model = GaussianDiffusion1D(
        n_input_channels=cfg['n_input_channels'],
        n_output_channels=cfg['n_output_channels'],
        d_model=cfg['d_model'],
        n_heads=cfg['n_heads'],
        n_layers=cfg['n_layers'],
        d_ff=cfg['d_ff'],
        dropout=cfg['dropout'],
        n_timesteps=cfg.get('n_timesteps', 1000),
        schedule=cfg.get('schedule', 'cosine'),
        predict_epsilon=cfg.get('predict_epsilon', True),
        n_inference_steps=cfg.get('n_inference_steps', 50),
    )
else:
    model = TCN(
        n_input_channels=cfg['n_input_channels'],
        n_output_channels=cfg['n_output_channels'],
        hidden_channels=cfg['hidden_channels'],
        n_blocks=cfg['n_blocks'],
        kernel_size=cfg['kernel_size'],
        dropout=cfg['dropout'],
    )

model.load_state_dict(ckpt['model_state_dict'])
model.to(DEVICE)
model.eval()

WINDOW_SIZE = int(ckpt.get('window_size', 100))
INPUT_INDICES = list(ckpt.get('input_indices', [6, 9, 10, 13, 16, 17]))
MOMENT_INDICES = list(ckpt.get('moment_indices', [3, 12, 14, 6, 13, 15]))
DOF_NAMES = list(ckpt.get('dof_names', ['hip_flexion', 'knee_angle', 'ankle_angle']))

# Right/left index split for unilateral-paired inference
H = len(INPUT_INDICES) // 2
INPUT_IDX_R = INPUT_INDICES[:H]
INPUT_IDX_L = INPUT_INDICES[H:]

ID_COLS = [
    'hip_flexion_r_moment',
    'knee_angle_r_moment',
    'ankle_angle_r_moment',
    'hip_flexion_l_moment',
    'knee_angle_l_moment',
    'ankle_angle_l_moment',
]

print('Model config:', cfg)
print('window_size:', WINDOW_SIZE)
print('input_indices:', INPUT_INDICES)
print('moment_indices:', MOMENT_INDICES)
print('DOF names:', DOF_NAMES)

In [ ]:
@torch.no_grad()
def causal_infer_one_side(pos_3: np.ndarray, vel_3: np.ndarray) -> np.ndarray:
    """
    pos_3, vel_3: (T, 3), already filtered, radians / rad/s.
    Returns: (T, 3) model output in N·m/kg.
    """
    x = np.concatenate([pos_3, vel_3], axis=1).astype(np.float32)  # (T, 6)
    n = x.shape[0]
    W = WINDOW_SIZE
    c_out = cfg['n_output_channels']

    if n < W:
        raise ValueError(f'Trial too short: n={n}, window={W}')

    pred = np.zeros((n, c_out), dtype=np.float64)

    def _fwd(start: int) -> np.ndarray:
        xw = np.ascontiguousarray(x[start:start + W].T)
        xt = torch.from_numpy(xw).unsqueeze(0).to(DEVICE)
        yw = model(xt).squeeze(0).detach().cpu().numpy().T  # (W, c_out)
        return yw

    # Bootstrap first window
    y0 = _fwd(0)
    pred[:W] = y0

    # Causal rolling: keep newest sample from each subsequent window
    for start in range(1, n - W + 1):
        yw = _fwd(start)
        pred[start + W - 1] = yw[W - 1]

    return pred.astype(np.float32)


def run_bilateral_inference(pos_full: np.ndarray, vel_full: np.ndarray) -> np.ndarray:
    pred_r = causal_infer_one_side(pos_full[:, INPUT_IDX_R], vel_full[:, INPUT_IDX_R])
    pred_l = causal_infer_one_side(pos_full[:, INPUT_IDX_L], vel_full[:, INPUT_IDX_L])
    return np.concatenate([pred_r, pred_l], axis=1)


def build_model_input_from_pkl(imu_dict: dict) -> np.ndarray:
    """Build full IK vector (T, len(IK_DOF_NAMES)) from MTw IK pickle channels (degrees)."""
    n = len(next(iter(imu_dict.values())))
    pos_deg = np.zeros((n, len(IK_DOF_NAMES)), dtype=np.float64)

    key_map = {
        'hip_flexion_r': 'hip_flexion_r',
        'knee_angle_r': 'knee_flexion_r',
        'ankle_angle_r': 'ankle_flexion_r',
        'hip_flexion_l': 'hip_flexion_l',
        'knee_angle_l': 'knee_flexion_l',
        'ankle_angle_l': 'ankle_flexion_l',
    }

    for ik_name, pkl_name in key_map.items():
        if pkl_name not in imu_dict:
            raise KeyError(f'Missing channel in pkl: {pkl_name}')
        idx = IK_DOF_NAMES.index(ik_name)
        pos_deg[:, idx] = np.asarray(imu_dict[pkl_name], dtype=np.float64)

    return pos_deg

In [ ]:
# Discover candidate trials
records = []
for subj_dir in sorted(IMU_IK_ROOT.glob('AB*')):
    if not subj_dir.is_dir():
        continue
    subject = subj_dir.name
    subj_opensim_dir = OPENSIM_ROOT / subject
    mass_kg = estimate_subject_mass_kg(subj_opensim_dir)

    for pkl_path in sorted(subj_dir.glob('*.pkl')):
        cond = filename_to_condition(pkl_path.stem)
        id_path, exo_folder = find_id_path(subject, cond)
        records.append({
            'subject': subject,
            'pkl_file': pkl_path.name,
            'condition': cond,
            'id_found': id_path is not None,
            'id_path': id_path,
            'exo_folder': exo_folder,
            'mass_kg': mass_kg,
        })

manifest_df = pd.DataFrame(records)
display(manifest_df)

print('Total IMU IK files:', len(manifest_df))
print('Matched with ID:', int(manifest_df['id_found'].sum()))
print('Unmatched (will skip):', int((~manifest_df['id_found']).sum()))

In [ ]:
# Run inference + comparison
TRIAL_DATA = {}
rows = []

for _, row in manifest_df.iterrows():
    if not row['id_found']:
        continue

    subject = row['subject']
    pkl_file = row['pkl_file']
    cond = row['condition']
    id_path = Path(row['id_path'])
    mass_kg = float(row['mass_kg']) if pd.notna(row['mass_kg']) else np.nan

    pkl_path = IMU_IK_ROOT / subject / pkl_file
    imu = pickle.load(open(pkl_path, 'rb'))
    pos_deg = build_model_input_from_pkl(imu)              # (T_imu, 23)
    pos_rad = np.deg2rad(pos_deg)

    # ID timeline defines fs for derivative/filter settings
    id_df = parse_opensim_table(id_path)
    t_id = id_df.index.to_numpy(dtype=np.float64)
    fs_hz = 1.0 / float(np.median(np.diff(t_id))) if len(t_id) > 2 else DEFAULT_FS_HZ

    # Requested filters: 6 Hz on angle, 15 Hz on velocity, 6 Hz on output (all causal, order 4)
    pos_f = causal_lpf_multichannel(pos_rad, fs_hz=fs_hz, cutoff_hz=ANGLE_CUTOFF_HZ, order=FILTER_ORDER)
    vel_raw = np.gradient(pos_f, 1.0 / fs_hz, axis=0)
    vel_f = causal_lpf_multichannel(vel_raw, fs_hz=fs_hz, cutoff_hz=VEL_CUTOFF_HZ, order=FILTER_ORDER)

    pred_nmpkg = run_bilateral_inference(pos_f, vel_f)
    pred_nmpkg_f = causal_lpf_multichannel(pred_nmpkg, fs_hz=fs_hz, cutoff_hz=OUT_CUTOFF_HZ, order=FILTER_ORDER)

    # Convert to N·m when mass is available
    pred_nm_f = pred_nmpkg_f * mass_kg if np.isfinite(mass_kg) else np.full_like(pred_nmpkg_f, np.nan)

    # Build ID matrix (N·m)
    id_nm = np.zeros((len(t_id), len(ID_COLS)), dtype=np.float64)
    for c, col in enumerate(ID_COLS):
        id_nm[:, c] = id_df[col].to_numpy(dtype=np.float64) if col in id_df.columns else np.nan

    # Align by index over common window (IMU pkl has no explicit timestamps)
    n_common = min(len(pred_nmpkg_f), len(id_nm))
    pred_nmpkg_c = pred_nmpkg_f[:n_common]
    pred_nm_c = pred_nm_f[:n_common]
    id_nm_c = id_nm[:n_common]
    t_c = t_id[:n_common]

    if np.isfinite(mass_kg):
        id_nmpkg_c = id_nm_c / mass_kg
    else:
        id_nmpkg_c = np.full_like(id_nm_c, np.nan)

    # Metrics in N·m/kg (primary) and N·m (if mass available)
    metrics = []
    for c in range(len(ID_COLS)):
        m = {}
        v_kg = np.isfinite(pred_nmpkg_c[:, c]) & np.isfinite(id_nmpkg_c[:, c])
        if v_kg.sum() > 1:
            e_kg = pred_nmpkg_c[v_kg, c] - id_nmpkg_c[v_kg, c]
            corr_kg = np.corrcoef(pred_nmpkg_c[v_kg, c], id_nmpkg_c[v_kg, c])[0, 1]
            m['rmse_nmpkg'] = float(np.sqrt(np.mean(e_kg ** 2)))
            m['r2_nmpkg'] = float(corr_kg ** 2)
        else:
            m['rmse_nmpkg'] = np.nan
            m['r2_nmpkg'] = np.nan

        v_nm = np.isfinite(pred_nm_c[:, c]) & np.isfinite(id_nm_c[:, c])
        if v_nm.sum() > 1:
            e_nm = pred_nm_c[v_nm, c] - id_nm_c[v_nm, c]
            corr_nm = np.corrcoef(pred_nm_c[v_nm, c], id_nm_c[v_nm, c])[0, 1]
            m['rmse_nm'] = float(np.sqrt(np.mean(e_nm ** 2)))
            m['r2_nm'] = float(corr_nm ** 2)
        else:
            m['rmse_nm'] = np.nan
            m['r2_nm'] = np.nan

        metrics.append(m)

    trial_key = f'{subject}::{cond}'
    TRIAL_DATA[trial_key] = {
        'subject': subject,
        'condition': cond,
        'pkl_file': pkl_file,
        'id_path': str(id_path),
        'mass_kg': mass_kg,
        't': t_c,
        'pred_nmpkg': pred_nmpkg_c,
        'pred_nm': pred_nm_c,
        'id_nmpkg': id_nmpkg_c,
        'id_nm': id_nm_c,
        'metrics': metrics,
    }

    rows.append({
        'trial': trial_key,
        'subject': subject,
        'condition': cond,
        'pkl_file': pkl_file,
        'mass_kg': mass_kg,
        'n_common': n_common,
        'mean_rmse_nmpkg': float(np.nanmean([m['rmse_nmpkg'] for m in metrics])),
        'mean_r2_nmpkg': float(np.nanmean([m['r2_nmpkg'] for m in metrics])),
        'mean_rmse_nm': float(np.nanmean([m['rmse_nm'] for m in metrics])),
        'mean_r2_nm': float(np.nanmean([m['r2_nm'] for m in metrics])),
    })

summary_df = pd.DataFrame(rows).sort_values(['subject', 'condition']).reset_index(drop=True)
display(summary_df)
print('Computed trials:', len(summary_df))

In [ ]:
# Channel-wise summary table
CHANNELS = [
    'hip_flexion_r', 'knee_angle_r', 'ankle_angle_r',
    'hip_flexion_l', 'knee_angle_l', 'ankle_angle_l',
]

detail_rows = []
for trial, d in TRIAL_DATA.items():
    for c, ch in enumerate(CHANNELS):
        m = d['metrics'][c]
        detail_rows.append({
            'trial': trial,
            'channel': ch,
            'rmse_nmpkg': m['rmse_nmpkg'],
            'r2_nmpkg': m['r2_nmpkg'],
            'rmse_nm': m['rmse_nm'],
            'r2_nm': m['r2_nm'],
        })

detail_df = pd.DataFrame(detail_rows)
display(detail_df)

print('Mean by channel (N·m/kg):')
display(detail_df.groupby('channel')[['rmse_nmpkg', 'r2_nmpkg']].mean().sort_index())

In [ ]:
# Interactive trial plot
if not TRIAL_DATA:
    raise RuntimeError('No trials available. Check path mapping and ID availability.')

trial_keys = sorted(TRIAL_DATA.keys())
trial_dd = widgets.Dropdown(options=trial_keys, value=trial_keys[0], description='Trial:')
unit_dd = widgets.Dropdown(options=['N·m/kg', 'N·m'], value='N·m/kg', description='Unit:')
out = widgets.Output()


def draw_trial(trial_key: str, unit: str):
    d = TRIAL_DATA[trial_key]
    t = d['t'] - d['t'][0]

    if unit == 'N·m/kg':
        y_pred = d['pred_nmpkg']
        y_id = d['id_nmpkg']
        rmse_key = 'rmse_nmpkg'
        r2_key = 'r2_nmpkg'
        ylab = 'N·m/kg'
    else:
        y_pred = d['pred_nm']
        y_id = d['id_nm']
        rmse_key = 'rmse_nm'
        r2_key = 'r2_nm'
        ylab = 'N·m'

    names = ['Hip R', 'Knee R', 'Ankle R', 'Hip L', 'Knee L', 'Ankle L']
    fig, axs = plt.subplots(3, 2, figsize=(14, 10), sharex=True)
    axs = axs.reshape(-1)

    for c in range(6):
        ax = axs[c]
        ax.plot(t, y_id[:, c], label='OpenSim ID', color='#1e88e5', lw=1.8)
        ax.plot(t, y_pred[:, c], label='Model', color='#e53935', lw=1.4, ls='--')
        m = d['metrics'][c]
        ax.set_title(f"{names[c]} | RMSE={m[rmse_key]:.3f}, R²={m[r2_key]:.3f}")
        ax.axhline(0.0, color='gray', lw=0.7, ls=':')
        ax.set_ylabel(ylab)
        ax.spines[['top', 'right']].set_visible(False)
        if c == 0:
            ax.legend(loc='upper right')

    axs[-2].set_xlabel('Time (s)')
    axs[-1].set_xlabel('Time (s)')
    fig.suptitle(f"{trial_key} | mass={d['mass_kg']:.2f} kg | filters: 6/15/6 Hz causal (order 4)", fontsize=12)
    fig.tight_layout()
    plt.show()


def redraw(*_):
    out.clear_output(wait=True)
    with out:
        draw_trial(trial_dd.value, unit_dd.value)

trial_dd.observe(redraw, names='value')
unit_dd.observe(redraw, names='value')

display(widgets.HBox([trial_dd, unit_dd]), out)
redraw()